This notebook summarizes number of genes per strain that have log_TPM > 1 in at least one condition and classfies whether the BGCs are expressed or not.

In [5]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

# Path containing subfolders for each strain
base_path = Path("/Users/annasve/Desktop/article_data/input/gene_expression_metadata")

# Media names to identify logTPM columns
media_keywords = ["DNPM", "ISP2", "MA", "SoyM", "TSB", "gluc", "gly", "malt"]
pattern = re.compile("|".join(media_keywords), re.IGNORECASE)

# Store results
strain_summary = []

for strain_folder in base_path.iterdir():
    if not strain_folder.is_dir():
        continue

    # Find log_tpm file(s)
    log_tpm_files = [f for f in strain_folder.glob("*CBI_overall*.csv")]
    if not log_tpm_files:
        continue

    # Usually one file per strain
    file_path = log_tpm_files[0]
    strain_name = strain_folder.name

    # Read file (try csv, fallback to tsv)
    try:
        df = pd.read_csv(file_path)
    except:
        df = pd.read_csv(file_path, sep="\t")

    # Identify expression columns by media name
    expr_cols = [c for c in df.columns if pattern.search(c)]
    if not expr_cols:
        continue

    # Count how many genes have logTPM > 1 in at least one condition
    expressed_genes = (df[expr_cols] > 1).any(axis=1).sum()
    total_genes = len(df)

    strain_summary.append({
        "Strain": strain_name,
        "Total_Genes": total_genes,
        "Expressed_Genes": expressed_genes,
        "Pct_Expressed": 100 * expressed_genes / total_genes
    })

# Combine into a summary dataframe
summary_df = pd.DataFrame(strain_summary)

# Compute overall statistics
summary_stats = summary_df["Pct_Expressed"].describe()

# Save outputs
summary_df.to_csv("expression_summary_by_strain.csv", index=False)

print("✅ Per-strain expression summary saved to 'expression_summary_by_strain.csv'")
print("\nOverall summary:")
print(summary_stats)


✅ Per-strain expression summary saved to 'expression_summary_by_strain.csv'

Overall summary:
count    132.000000
mean      99.260319
std        0.757508
min       96.534226
25%       98.796268
50%       99.573717
75%       99.791243
max      100.000000
Name: Pct_Expressed, dtype: float64


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# -----------------------------------------------------------
# USER INPUT
# -----------------------------------------------------------
base_path = Path("/Users/annasve/Desktop/article_data/input/gene_expression_metadata")

# media identifiers that appear in column names
media_keywords = ["DNPM", "ISP2", "MA", "SoyM", "TSB", "gluc", "gly", "malt"]

# -----------------------------------------------------------
# HELPER FUNCTIONS
# -----------------------------------------------------------
def detect_medium(col_name, keywords):
    col_name_clean = str(col_name).lower().replace("-", "_")
    for m in keywords:
        pattern = rf"(?:^|_|-){m.lower()}(?:_|-|$)"
        if re.search(pattern, col_name_clean):
            return m
    return "Unknown"

def summarize_media(group):
    expressed_media = group.loc[group["Expressed"] == True, "Medium"].dropna().unique()
    if len(expressed_media) == 0:
        return "not_expressed"
    return ", ".join(sorted(expressed_media))

def normalize_bgc_type(label):
    """Normalize multi-part BGC type strings so that order doesn't matter."""
    if pd.isna(label):
        return label
    # Split by underscores
    parts = label.split("_")
    # Sort alphabetically and rejoin with underscores
    parts_sorted = sorted(parts)
    return "_".join(parts_sorted)

# -----------------------------------------------------------
# INITIAL SETUP
# -----------------------------------------------------------
bgc_results = []              # per-sample summaries
bgc_status_detailed = []      # detailed per-BGC expression status
skip_log = []                 # debug: why folders were skipped?
processed_strains = []        # debug: which strains processed?

pattern = re.compile("|".join(media_keywords), re.IGNORECASE)

# -----------------------------------------------------------
# MAIN LOOP
# -----------------------------------------------------------
for strain_folder in base_path.iterdir():
    if not strain_folder.is_dir():
        continue

    strain = strain_folder.name

    # find file(s)
    files = list(strain_folder.glob("*CBI_overall*.csv"))
    if not files:
        skip_log.append({"Strain": strain, "Reason": "No *CBI_overall*.csv found"})
        continue

    f = files[0]
    print(f"\n🧬 Processing strain: {strain}")

    # --- Load data ---
    try:
        df = pd.read_csv(f)
    except Exception as e1:
        try:
            df = pd.read_csv(f, sep="\t")
        except Exception as e2:
            skip_log.append({"Strain": strain, "Reason": f"Could not read CSV/TSV: {e1} | {e2}"})
            continue

    # --- Normalize column names ---
    df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

    # --- Find cluster product column ---
    cluster_prod_cols = [c for c in df.columns if c.startswith("Cluster_Product")]
    if not cluster_prod_cols:
        skip_log.append({"Strain": strain, "Reason": "No Cluster_Product* column found"})
        continue
    cluster_col = cluster_prod_cols[0]

    required_cols = {"Region_Nr", cluster_col, "Is_Core_Gene"}
    if not required_cols.issubset(df.columns):
        missing = sorted(list(required_cols - set(df.columns)))
        skip_log.append({"Strain": strain, "Reason": f"Missing required columns: {missing}"})
        continue

    # --- Ensure Is_Core_Gene is boolean ---
    df["Is_Core_Gene"] = df["Is_Core_Gene"].astype(str).str.lower().isin(["true", "1", "yes"])

    # --- Identify expression columns ---
    expr_cols = [c for c in df.columns if pattern.search(c)]
    if not expr_cols:
        skip_log.append({"Strain": strain, "Reason": "No expression columns matched media keywords"})
        continue

    processed_strains.append(strain)

    # -----------------------------------------------------------
    # Normalize cluster_col BEFORE building BGC_ID
    # -----------------------------------------------------------
    df[cluster_col] = df[cluster_col].apply(normalize_bgc_type)

    # -----------------------------------------------------------
    # Build BGC_ID
    # -----------------------------------------------------------
    df["BGC_ID"] = df["Region_Nr"].astype(str) + "_" + df[cluster_col].astype(str) + "_" + strain

    all_bgc_activity = []  # collect per-medium per-BGC status for the strain

    # -----------------------------------------------------------
    # iterate over expression columns
    # -----------------------------------------------------------
    for col in expr_cols:
        medium = detect_medium(col, media_keywords)
        if medium == "Unknown":
            continue

        # compute 75th percentile threshold per sample
        threshold_75th = df[col].quantile(0.75)

        results = []
        for bgc_id, bgc_data in df.groupby("BGC_ID"):
            core_genes = bgc_data[bgc_data["Is_Core_Gene"]]
            if core_genes.empty:
                continue

            frac_high = (core_genes[col] > threshold_75th).mean()
            expressed = frac_high > 0.5

            results.append({
                "BGC_ID": bgc_id,
                "Frac_High_Expr": frac_high,
                "Expressed": expressed
            })

        n_total = len(results)
        n_expressed = sum(r["Expressed"] for r in results) if n_total > 0 else 0
        pct_expressed = 100 * n_expressed / n_total if n_total > 0 else 0

        print(f"   • {medium:<6} → total={n_total:3d}, expressed={n_expressed:3d} ({pct_expressed:.1f}%)")

        if results:
            bgc_df = pd.DataFrame(results)
            bgc_df["Strain"] = strain
            bgc_df["Medium"] = medium
            bgc_df["Sample"] = col

            all_bgc_activity.append(bgc_df)

            bgc_results.append({
                "Strain": strain,
                "Medium": medium,
                "Sample": col,
                "Total_BGCs": n_total,
                "Expressed_BGCs": n_expressed,
                "Pct_Expressed": pct_expressed,
                "Threshold_75th": threshold_75th
            })

            # detailed per BGC + medium
            for _, row in bgc_df.iterrows():
                bgc_status_detailed.append({
                    "Strain": strain,
                    "Medium": medium,
                    "BGC_ID": row["BGC_ID"],
                    "Expressed": bool(row["Expressed"]),
                    "Frac_High_Expr": float(row["Frac_High_Expr"])
                })

    # -----------------------------------------------------------
    # PER-STRAIN SUMMARY (across all media): Expressed in ANY medium?
    # -----------------------------------------------------------
    if all_bgc_activity:
        all_bgc_activity = pd.concat(all_bgc_activity, ignore_index=True)

        bgc_across_conditions = (
            all_bgc_activity.groupby("BGC_ID")["Expressed"]
            .any()
            .reset_index()
            .rename(columns={"Expressed": "Expressed_Any"})
        )

        total_bgcs = bgc_across_conditions.shape[0]
        expressed_bgcs = int(bgc_across_conditions["Expressed_Any"].sum())
        silent_bgcs = total_bgcs - expressed_bgcs

        # write per-strain summary
        with open("bgc_expression_per_strain_summary.csv", "a") as fsum:
            fsum.write(f"{strain},{total_bgcs},{expressed_bgcs},{silent_bgcs}\n")

        # add "All" rows (for later; you can drop these downstream if needed)
        for _, row in bgc_across_conditions.iterrows():
            bgc_status_detailed.append({
                "Strain": strain,
                "Medium": "All",
                "BGC_ID": row["BGC_ID"],
                "Expressed": bool(row["Expressed_Any"]),
                "Frac_High_Expr": np.nan
            })

    print(f"✅ Finished strain {strain}. Processed media: {sorted(set([detect_medium(c, media_keywords) for c in expr_cols]))}")

# -----------------------------------------------------------
# COMPILE AND SAVE RESULTS
# -----------------------------------------------------------
bgc_summary = pd.DataFrame(bgc_results)
bgc_status_detailed_df = pd.DataFrame(bgc_status_detailed)
skip_df = pd.DataFrame(skip_log)

# save debug info
skip_df.to_csv("skipped_folders_debug.csv", index=False)

print("\n====================")
print(f"Processed strains: {len(processed_strains)}")
print(f"Skipped strains:   {len(skip_df)}")
if not skip_df.empty:
    print("\nSkip reasons:")
    print(skip_df["Reason"].value_counts())
    print("\n❌ Skipped strains (with reasons):")
    for _, r in skip_df.sort_values(["Reason", "Strain"]).iterrows():
        print(f"{r['Strain']}\t{r['Reason']}")
    print("\nSaved: skipped_folders_debug.csv")
print("====================\n")

# save main outputs
if not bgc_summary.empty:
    bgc_by_medium = (
        bgc_summary.groupby("Medium")["Pct_Expressed"]
        .agg(["mean", "median", "std", "count"])
        .reset_index()
    )

    bgc_summary.to_csv("bgc_expression_per_sample.csv", index=False)
    bgc_by_medium.to_csv("bgc_expression_summary_by_medium.csv", index=False)
    bgc_status_detailed_df.to_csv("bgc_expression_status_detailed.csv", index=False)

    print("✅ Results saved:")
    print(" - bgc_expression_per_sample.csv")
    print(" - bgc_expression_summary_by_medium.csv")
    print(" - bgc_expression_per_strain_summary.csv")
    print(" - bgc_expression_status_detailed.csv")
else:
    print("⚠️ No BGC/sample summaries produced (bgc_summary is empty).")

# -----------------------------------------------------------
# OPTIONAL: One row per BGC showing media where expressed (dropping 'All')
# -----------------------------------------------------------
if not bgc_status_detailed_df.empty:
    bgc_status_no_all = bgc_status_detailed_df[bgc_status_detailed_df["Medium"] != "All"].copy()

    # If you still want a cleaned BGC string column (optional)
    bgc_status_no_all["BGC_clean"] = (
        bgc_status_no_all["BGC_ID"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.replace(", ", "_", regex=False)
    )

    bgc_media_summary = (
        bgc_status_no_all
        .groupby(["Strain", "BGC_clean"], as_index=False)
        .apply(lambda g: pd.Series({
            "media_expressed": summarize_media(g),
            "ever_expressed": bool((g["Expressed"] == True).any())
        }))
    )

    bgc_media_summary.to_csv("bgc_media_expression_per_bgc.csv", index=False)
    print("\n✅ Saved: bgc_media_expression_per_bgc.csv")



🧬 Processing strain: NBC_01734
   • DNPM   → total= 15, expressed=  1 (6.7%)
   • ISP2   → total= 15, expressed=  0 (0.0%)
   • MA     → total= 15, expressed=  0 (0.0%)
   • SoyM   → total= 15, expressed=  1 (6.7%)
   • TSB    → total= 15, expressed=  2 (13.3%)
   • gluc   → total= 15, expressed=  2 (13.3%)
   • gly    → total= 15, expressed=  0 (0.0%)
   • malt   → total= 15, expressed=  2 (13.3%)
✅ Finished strain NBC_01734. Processed media: ['DNPM', 'ISP2', 'MA', 'SoyM', 'TSB', 'gluc', 'gly', 'malt']

🧬 Processing strain: NBC_01392
   • DNPM   → total= 20, expressed=  0 (0.0%)
   • DNPM   → total= 20, expressed=  0 (0.0%)
   • ISP2   → total= 20, expressed=  3 (15.0%)
   • ISP2   → total= 20, expressed=  3 (15.0%)
   • MA     → total= 20, expressed=  2 (10.0%)
   • MA     → total= 20, expressed=  2 (10.0%)
   • SoyM   → total= 20, expressed=  0 (0.0%)
   • SoyM   → total= 20, expressed=  0 (0.0%)
   • TSB    → total= 20, expressed=  3 (15.0%)
   • TSB    → total= 20, expressed=  3 